# SmolVLA fine-tuned on dataset v1 (200 demonstrations), Google Colab (free T4), resumable

The same training as `train_smolvla_v1_kaggle.ipynb`:
- `lerobot/smolvla_base` fine-tuned on `Kimaa6/icub_gardener_v1`, with ACT v1's split (`configs/act_split.json`:
  180 training / 20 validation episodes) and training seed 1000;
- the 3 cameras, the 12 joints and the instruction sentence; no task-ID vector.

**Why this notebook is different:** free Colab sessions stop without warning, after some hours or when the tab sits
idle. So training runs on Colab's own disk, and a background copier saves every finished checkpoint to your Google
Drive. **If the session stops, open the notebook again and run all cells: it resumes from the last checkpoint on
Drive.** At most one checkpoint interval of work is lost (a tenth of the run).

**Before running:**
1. **Runtime, Change runtime type: T4 GPU.**
2. **Secret:** the key icon on the left, add `HF_TOKEN` (read access to the dataset) and switch on "Notebook access".
3. **Drive space:** at least **8 GB free** in your Google Drive. Everything goes to `MyDrive/icub_gardener_smolvla/`.
4. **Runtime, Run all.** Accept the Drive access prompt in cell 1.
5. **Keep the tab open, and click in it now and then:** an idle tab is disconnected sooner.

**The first time,** cell 6 measures the GPU memory and the seconds per step, and saves the plan to Drive. Note those
numbers down. Later sessions reuse the saved plan, so every piece of the run has the same steps and learning
rate schedule.

**At the end** (cell 9), `MyDrive/icub_gardener_smolvla/` holds:
- 5 checkpoints (`checkpoints/<step>/pretrained_model`, about 0.9 GB each);
- `smolvla_val.txt`, which names the 3 with the lowest validation loss. Download those 3 folders from Drive.

In [ ]:
# 1. GPU, versions, Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
from google.colab import drive
drive.mount("/content/drive")
import os
D = "/content/drive/MyDrive/icub_gardener_smolvla"     # what survives the session
W = "/content"                                         # fast local disk, lost when the session ends
os.makedirs(f"{D}/checkpoints", exist_ok=True)
!df -h /content | tail -1

In [ ]:
# 2. Code: the public repo, main branch. The commit is printed so the run can be traced.
!rm -rf /content/icub-gardener
!git clone -q -b main https://github.com/kaarima/icub-gardener.git /content/icub-gardener
!cd /content/icub-gardener && git log --oneline -1 | tee -a /content/drive/MyDrive/icub_gardener_smolvla/commits.txt

In [ ]:
# 3. LeRobot 0.4.4 with its SmolVLA extra (it pins transformers to 4.x; 5.x does not work with LeRobot 0.4.4).
# '!' lines do not stop the run when they fail, so the import test runs with check=True: a failure stops here.
import subprocess
!pip install -q "lerobot[smolvla]==0.4.4"
subprocess.run(["python", "-c", "import lerobot, torch, transformers, lerobot.configs.train, lerobot.policies.smolvla.modeling_smolvla; "
                "print('lerobot', lerobot.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, "
                "'| cuda', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported()); "
                "assert transformers.__version__.startswith('4.'), transformers.__version__; "
                "assert torch.cuda.is_available(), 'no GPU: Runtime, Change runtime type, T4 GPU'"], check=True)

In [ ]:
# 4. Dataset v1 from the Hub (private: needs the HF_TOKEN secret)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!hf download Kimaa6/icub_gardener_v1 --repo-type dataset --local-dir /content/gardener_v1 --quiet
!du -sh /content/gardener_v1

In [ ]:
# 5. Checks before spending GPU hours: 200 episodes, ACT v1's split, the 6 instructions as sentences, one frame decodes.
import json
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

REPO, ROOT = "Kimaa6/icub_gardener_v1", "/content/gardener_v1"
split = json.load(open("/content/icub-gardener/configs/act_split.json"))
meta = LeRobotDatasetMetadata(REPO, root=ROOT)
train, val = set(split["train"]), set(split["val"])
assert meta.total_episodes == 200, meta.total_episodes
assert len(train) == 180 and len(val) == 20 and not train & val and train | val == set(range(200)), (len(train), len(val))
tasks = sorted(meta.tasks.index)
assert tasks == sorted(["Water the left plant.", "Water the right plant.", "Give the left plant just a little water.",
                        "Give the right plant just a little water.", "Give the left plant a lot of water.",
                        "Give the right plant a lot of water."]), tasks
item = LeRobotDataset(REPO, root=ROOT, episodes=[split["val"][0]])[0]
print({k: tuple(v.shape) for k, v in item.items() if hasattr(v, "shape")}, repr(item["task"]))
print(f"OK: {meta.total_episodes} episodes, {meta.total_frames} frames; 180 train / 20 val, none in both; tasks: {tasks}")

In [ ]:
# 6. The plan: measured once, then saved on Drive and reused by every later session.
# Measurement: 60 steps at batch 32, then 16 if 32 runs out of memory; peak GPU memory (nvidia-smi every 0.5 s) and
# seconds per step (LeRobot's updt_s + data_s, after warm-up).
# Plan: the batch that fits, and the steps that 9 h of training allow, at most 20,000 (the same rule as the Kaggle
# notebook, so the two give the same run). A checkpoint every tenth of the run, of which every second one is kept.
import re, shutil, subprocess, threading, time
REPO_DIR = "/content/icub-gardener"
base = dict(os.environ, DATASET=ROOT, REPO_ID=REPO, SPLIT="configs/act_split.json", WORKERS="2", LOG="10")
PLAN = f"{D}/smolvla_plan.json"

def measure(batch):
    out = f"{W}/smolvla_measure_{batch}"
    shutil.rmtree(out, ignore_errors=True)
    peak, done = [0], [False]
    def poll():
        while not done[0]:
            r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True)
            peak[0] = max(peak[0], int(r.stdout.split()[0]))
            time.sleep(0.5)
    t = threading.Thread(target=poll); t.start()
    env = dict(base, OUT=out, STEPS="60", BATCH=str(batch), SAVE="100000")
    p = subprocess.run(["bash", "scripts/train_smolvla.sh"], cwd=REPO_DIR, env=env, capture_output=True, text=True)
    done[0] = True; t.join()
    log = p.stdout + p.stderr
    open(f"{D}/smolvla_measure_{batch}.log", "w").write(log)
    shutil.rmtree(out, ignore_errors=True)
    if p.returncode != 0:
        oom = "out of memory" in log.lower()
        print(f"batch {batch}: {'OUT OF MEMORY' if oom else 'FAILED'}, peak {peak[0]} MiB")
        if not oom:
            print(log[-3000:])
            raise SystemExit(f"measurement at batch {batch} failed for another reason than memory: see above")
        return None
    s = [float(a) + float(b) for a, b in re.findall(r"updt_s:([0-9.]+) data_s:([0-9.]+)", log)][2:]
    params = [l for l in log.splitlines() if "params" in l.lower()][:2]
    sec = sum(s) / len(s)
    print(f"batch {batch}: peak GPU memory {peak[0]} MiB, {sec:.2f} s per step ({1 / sec:.2f} steps/s); {params}")
    return dict(batch=batch, peak_mib=peak[0], sec_per_step=sec)

if os.path.exists(PLAN):
    plan = json.load(open(PLAN))
    print("plan from Drive (measured in an earlier session):", plan)
else:
    m = measure(32) or measure(16)
    assert m, "neither batch 32 nor 16 fits in GPU memory"
    steps = min(20000, int(9 * 3600 / m["sec_per_step"] / 1000) * 1000)
    assert steps >= 5000, f"only {steps} steps fit in 9 h: too slow for a useful run, stop and rethink"
    plan = dict(m, steps=steps, save=steps // 10, keep_every=steps // 5,
                hours=round(steps * m["sec_per_step"] / 3600, 1), samples=steps * m["batch"],
                passes=round(steps * m["batch"] / sum(meta.episodes["length"][e] for e in train), 2),
                gpu=subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
    json.dump(plan, open(PLAN, "w"))
    print("PLAN (saved to Drive):", plan)
print("for comparison, ACT v1: 80,000 steps x batch 8 = 640,000 samples")

In [ ]:
# 7. Train, or resume from the newest checkpoint on Drive.
# A background copier puts every finished checkpoint on Drive (a checkpoint is finished once LeRobot points its
# 'last' link at it). On Drive it keeps: the optimizer state of the newest checkpoint only (needed to resume), and
# the model of every fifth of the run plus the newest. The full log is appended to Drive.
import glob, os, shutil, subprocess, threading, time
LOCAL, DRIVE_CK = f"{W}/smolvla", f"{D}/checkpoints"

def finished_local():
    last = f"{LOCAL}/checkpoints/last"
    if not os.path.islink(last):
        return []
    newest = os.path.basename(os.readlink(last))
    return [c for c in sorted(glob.glob(f"{LOCAL}/checkpoints/0*")) if os.path.basename(c) <= newest]

def on_drive():
    return sorted(c for c in glob.glob(f"{DRIVE_CK}/0*") if os.path.exists(f"{c}/.complete"))

def sync():
    for c in finished_local():
        dst = f"{DRIVE_CK}/{os.path.basename(c)}"
        if os.path.exists(f"{c}/.on_drive") or os.path.exists(f"{dst}/.complete"):
            continue                                      # copied already (and maybe thinned out since)
        shutil.rmtree(dst + ".partial", ignore_errors=True)
        shutil.copytree(c, dst + ".partial")
        shutil.rmtree(dst, ignore_errors=True)
        os.rename(dst + ".partial", dst)
        open(f"{dst}/.complete", "w").close()
        open(f"{c}/.on_drive", "w").close()
        print(f"saved to Drive: checkpoint {os.path.basename(c)}", flush=True)
    done = on_drive()
    for c in done[:-1]:                                   # older than the newest
        shutil.rmtree(f"{c}/training_state", ignore_errors=True)
        if int(os.path.basename(c)) % plan["keep_every"]:
            shutil.rmtree(c, ignore_errors=True)

done = on_drive()
final = f"{DRIVE_CK}/{plan['steps']:06d}"
env = dict(base, OUT=LOCAL, STEPS=str(plan["steps"]), BATCH=str(plan["batch"]), SAVE=str(plan["save"]), LOG="100")
if os.path.exists(f"{final}/.complete"):
    print("training already finished:", [os.path.basename(c) for c in done])
else:
    if done:                                              # resume: the newest checkpoint back to the local disk
        start = os.path.basename(done[-1])
        assert os.path.isdir(f"{done[-1]}/training_state"), f"checkpoint {start} has no optimizer state to resume from"
        shutil.rmtree(LOCAL, ignore_errors=True)
        shutil.copytree(done[-1], f"{LOCAL}/checkpoints/{start}")
        open(f"{LOCAL}/checkpoints/{start}/.on_drive", "w").close()
        os.symlink(start, f"{LOCAL}/checkpoints/last")
        cmd = ["lerobot-train", f"--config_path={LOCAL}/checkpoints/{start}/pretrained_model/train_config.json", "--resume=true"]
        print("RESUMING from step", int(start))
    else:
        shutil.rmtree(LOCAL, ignore_errors=True)
        cmd = ["bash", "scripts/train_smolvla.sh"]
        print("STARTING:", plan)
    stop = [False]
    def copier():
        while not stop[0]:
            try:
                sync()
            except Exception as e:                        # a Drive hiccup must not kill training; retried next time
                print("Drive copy will be retried:", e, flush=True)
            time.sleep(60)
    t = threading.Thread(target=copier, daemon=True); t.start()
    n = 0
    with open(f"{D}/smolvla_train.log", "a") as log:
        log.write(f"\n===== {time.ctime()} {' '.join(cmd)}\n")
        p = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line); log.flush()
            n += "step:" in line
            if (line.startswith("SmolVLA:") or "Checkpoint" in line or "rror" in line or "Traceback" in line
                    or "params" in line.lower() or ("step:" in line and n % 10 == 1)):
                print(line, end="", flush=True)
    code_ = p.wait()
    stop[0] = True; t.join()
    sync()
    assert code_ == 0, f"training failed (exit {code_}): see the end of {D}/smolvla_train.log"
done = on_drive()
assert os.path.exists(f"{final}/.complete"), f"the final checkpoint is not on Drive: {done}"
assert len(done) == 5, f"expected 5 kept checkpoints on Drive, found {[os.path.basename(c) for c in done]}"
print("training done; on Drive:", [os.path.basename(c) for c in done])

In [ ]:
# 8. Validation and training loss for the 5 kept checkpoints, on ACT v1's split. For SmolVLA this is its own
# flow-matching loss with fixed noise: it ranks its checkpoints, it is not comparable with ACT's numbers.
!cd /content/icub-gardener && python scripts/val_loss.py /content/drive/MyDrive/icub_gardener_smolvla/checkpoints/0*/pretrained_model \
   --root /content/gardener_v1 --repo-id Kimaa6/icub_gardener_v1 --split configs/act_split.json \
   2>&1 | grep -v Warn | tee /content/drive/MyDrive/icub_gardener_smolvla/smolvla_val.txt

In [ ]:
# 9. The 3 checkpoints with the lowest validation loss: download these folders from Drive for the dev-seed check.
import re
rows = re.findall(r"checkpoints/(\d+)/pretrained_model: validation ([0-9.]+)", open(f"{D}/smolvla_val.txt").read())
assert len(rows) == 5, f"expected a validation loss for 5 checkpoints, got {rows}: see smolvla_val.txt"
best = sorted(rows, key=lambda r: float(r[1]))[:3]
open(f"{D}/smolvla_best3.txt", "w").write("\n".join(f"{s} validation {v}" for s, v in best) + "\n")
print("download from MyDrive/icub_gardener_smolvla/checkpoints/:", [s for s, _ in best])
!du -sh /content/drive/MyDrive/icub_gardener_smolvla/checkpoints/*